# Data Quality — Rerun Failed Checks Notebook

**Use this notebook to rerun any checks that failed or errored in a previous run.**

- Looks up all `FAIL` or `ERROR` results for a given date (defaults to today)
- Re-executes the DAX for each failed check
- Writes new results with a fresh `run_id` — the original failure rows are preserved for audit history

Set `TARGET_DATE` below to rerun failures from a specific date, or leave as `None` to use today.

In [ ]:
# -- Configuration -------------------------------------------------------------
# Load shared settings from config.py in Lakehouse.
from datetime import datetime, timezone
import uuid
import time
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

TARGET_DATE = None  # Set to a date string like "2026-03-21" to rerun failures from that date, or None for today

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
now_utc = datetime.now(timezone.utc)
RUN_DATE = now_utc.date()
RUN_TIMESTAMP = now_utc.replace(tzinfo=None)
RUN_ID = str(uuid.uuid4())
LOOKUP_DATE = TARGET_DATE if TARGET_DATE else str(RUN_DATE)

print(f"Looking for failures on: {LOOKUP_DATE}")
print(f"New Run ID: {RUN_ID}")

In [ ]:
from IPython.display import display

# ── Load failed/errored checks for the target date ───────────────────
failures_df = spark.sql(f"""
    SELECT v.check_name, v.model_name, v.workspace_id, v.dataset_id, v.run_timestamp, v.status, v.error_message,
           r.workspace_name, r.dataset_name, r.dax_expression, r.run_frequency, r.fail_delta_pct_threshold
    FROM {FULL_SCHEMA}.validation_results v
    INNER JOIN {FULL_SCHEMA}.check_registry r
      ON v.check_name = r.check_name
     AND v.workspace_id = r.workspace_id
     AND v.dataset_id = r.dataset_id
    WHERE v.run_date = '{LOOKUP_DATE}'
      AND v.status IN ('FAIL', 'ERROR')
      AND r.is_active = true
    ORDER BY v.check_name, v.model_name, v.run_timestamp DESC
""").toPandas()

if len(failures_df) == 0:
    print(f"✓  No failures or errors found for {LOOKUP_DATE} — nothing to rerun")
else:
    # Deduplicate: keep latest failure per identity key
    failures_df = failures_df.drop_duplicates(
        subset=['check_name', 'workspace_id', 'dataset_id'], keep='first'
    )

    missing_ids = failures_df[
        failures_df["workspace_id"].isna()
        | failures_df["dataset_id"].isna()
        | (failures_df["workspace_id"].astype(str).str.strip() == "")
        | (failures_df["dataset_id"].astype(str).str.strip() == "")
    ]
    if len(missing_ids) > 0:
        raise RuntimeError("workspace_id/dataset_id are required for rerun_failed rows. Update registry first.")

    invalid_thresholds = failures_df[
        failures_df["fail_delta_pct_threshold"].isna()
        | (failures_df["fail_delta_pct_threshold"] < 0)
    ]
    if len(invalid_thresholds) > 0:
        raise RuntimeError(
            "Rerun-failed rows must have non-negative fail_delta_pct_threshold values.\n"
            + invalid_thresholds[["check_name", "model_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold"]].to_string(index=False)
        )

    print(f"Found {len(failures_df)} failed/errored check(s) to rerun:\n")
    display(failures_df[["check_name", "model_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold", "run_timestamp", "status", "error_message"]])

In [ ]:
# -- Rerun failed checks with Retry, Type Validation, Timeout -----------------
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, TimestampType

# Validate sempy is available
try:
    import sempy.fabric as sempy
except ImportError:
    raise RuntimeError(
        "sempy library not available. This notebook must run in Microsoft Fabric. "
        "Visit: https://learn.microsoft.com/en-us/python/api/sempy/"
    )

if len(failures_df) == 0:
    print("Nothing to rerun.")
else:
    affected_checks_escaped = [c.replace("'", "''") for c in failures_df["check_name"].unique()]
    affected_checks = "','".join(affected_checks_escaped)

    baseline_cfg_df = spark.sql(f"""
        SELECT
            check_name,
            COALESCE(baseline_mode, 'MODEL') AS baseline_mode,
            static_baseline_value
        FROM {FULL_SCHEMA}.check_baseline_config
        WHERE check_name IN ('{affected_checks}')
    """).toPandas()

    model_baselines_df = spark.sql(f"""
        SELECT
            check_name,
            model_name AS baseline_model,
            workspace_id AS baseline_workspace_id,
            dataset_id AS baseline_dataset_id,
            workspace_name AS baseline_workspace,
            dataset_name AS baseline_dataset,
            dax_expression AS baseline_dax
        FROM {FULL_SCHEMA}.check_registry
        WHERE check_name IN ('{affected_checks}')
          AND is_active = true
          AND is_baseline = true
    """).toPandas()

    expected_checks = sorted(set(failures_df["check_name"].unique()))

    cfg_map = {}
    for check_name in expected_checks:
        row = baseline_cfg_df[baseline_cfg_df["check_name"] == check_name]
        if len(row) == 0:
            cfg_map[check_name] = {"baseline_mode": "MODEL", "static_baseline_value": None}
        else:
            cfg_map[check_name] = {
                "baseline_mode": str(row.iloc[0]["baseline_mode"]).strip().upper(),
                "static_baseline_value": row.iloc[0]["static_baseline_value"]
            }

    invalid_modes = [c for c in expected_checks if cfg_map[c]["baseline_mode"] not in ["MODEL", "STATIC"]]
    if len(invalid_modes) > 0:
        raise RuntimeError(
            "Invalid baseline_mode for check_name(s): " + ", ".join(invalid_modes)
        )

    static_missing = [
        c for c in expected_checks
        if cfg_map[c]["baseline_mode"] == "STATIC" and cfg_map[c]["static_baseline_value"] is None
    ]
    if len(static_missing) > 0:
        raise RuntimeError(
            "STATIC baseline_mode requires static_baseline_value for check_name(s): "
            + ", ".join(static_missing)
        )

    model_checks = [c for c in expected_checks if cfg_map[c]["baseline_mode"] == "MODEL"]
    if len(model_checks) > 0:
        model_baseline_counts = model_baselines_df.groupby("check_name").size() if len(model_baselines_df) > 0 else None
        missing_model_baselines = []
        duplicate_model_baselines = []
        for check_name in model_checks:
            count = 0 if model_baseline_counts is None or check_name not in model_baseline_counts else int(model_baseline_counts[check_name])
            if count == 0:
                missing_model_baselines.append(check_name)
            elif count > 1:
                duplicate_model_baselines.append(check_name)

        if len(missing_model_baselines) > 0:
            raise RuntimeError(
                "Baseline invariant violation: no active baseline row found for MODEL check_name(s): "
                + ", ".join(missing_model_baselines)
            )
        if len(duplicate_model_baselines) > 0:
            raise RuntimeError(
                "Baseline invariant violation: multiple active baseline rows found for MODEL check_name(s): "
                + ", ".join(duplicate_model_baselines)
            )

    RESULT_BATCH_SCHEMA = StructType([
        StructField("run_id", StringType(), True),
        StructField("run_date", DateType(), True),
        StructField("run_timestamp", TimestampType(), True),
        StructField("check_name", StringType(), True),
        StructField("model_name", StringType(), True),
        StructField("workspace_id", StringType(), True),
        StructField("dataset_id", StringType(), True),
        StructField("workspace_name", StringType(), True),
        StructField("dataset_name", StringType(), True),
        StructField("result_value", DoubleType(), True),
        StructField("baseline_model", StringType(), True),
        StructField("baseline_value", DoubleType(), True),
        StructField("absolute_delta", DoubleType(), True),
        StructField("delta_pct", DoubleType(), True),
        StructField("fail_delta_pct_threshold", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
    ])

    # Helper: Call sempy.evaluate_dax across sempy versions.
    def evaluate_dax_compat(dataset, workspace, dax):
        try:
            return sempy.evaluate_dax(dataset=dataset, workspace=workspace, dax_string=dax)
        except TypeError as exc:
            if "unexpected keyword argument 'dax_string'" not in str(exc):
                raise
            return sempy.evaluate_dax(dataset=dataset, workspace=workspace, dax=dax)

    # Helper: Execute a single DAX with timeout protection
    def evaluate_dax_with_timeout(dataset, workspace, dax, timeout_seconds=DAX_TIMEOUT_SECONDS):
        executor = ThreadPoolExecutor(max_workers=1)
        future = executor.submit(evaluate_dax_compat, dataset, workspace, dax)
        try:
            return future.result(timeout=timeout_seconds), None
        except FuturesTimeoutError:
            future.cancel()
            return None, f"DAX execution timed out after {timeout_seconds}s"
        except Exception as e:
            return None, str(e)
        finally:
            executor.shutdown(wait=False, cancel_futures=True)

    # Helper: Retry with exponential backoff
    def evaluate_dax_with_retry(dataset, workspace, dax):
        for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
            result_df, error_msg = evaluate_dax_with_timeout(dataset, workspace, dax)
            if error_msg is None:
                return result_df, None
            if attempt < MAX_RETRY_ATTEMPTS:
                delay = INITIAL_RETRY_DELAY ** attempt
                time.sleep(delay)
            else:
                return None, error_msg
        return None, "Max retries exceeded"

    # Helper: Extract numeric value with type validation
    def extract_numeric_value(result_df):
        if result_df is None or len(result_df) == 0:
            return None, "DAX returned empty result"
        try:
            val = result_df.iloc[0, 0]
            if isinstance(val, (int, float)):
                return float(val), None
            if isinstance(val, str):
                try:
                    return float(val), None
                except ValueError:
                    return None, f"DAX returned non-numeric string: {val}"
            if val is None:
                return None, "DAX returned NULL"
            return None, f"DAX returned unsupported type: {type(val).__name__}"
        except Exception as e:
            return None, f"Failed to extract value: {str(e)[:200]}"

    # Helper: MERGE batch for idempotent writes
    def merge_results_batch(batch_rows):
        if not batch_rows:
            return

        batch_df = spark.createDataFrame(batch_rows, schema=RESULT_BATCH_SCHEMA)
        batch_df.createOrReplaceTempView("dq_rerun_failed_batch")

        spark.sql(f"""
        MERGE INTO {FULL_SCHEMA}.validation_results t
        USING dq_rerun_failed_batch s
          ON t.run_id = s.run_id
         AND t.check_name = s.check_name
         AND t.workspace_id = s.workspace_id
         AND t.dataset_id = s.dataset_id
        WHEN MATCHED THEN UPDATE SET
          t.run_date = s.run_date,
          t.run_timestamp = s.run_timestamp,
          t.model_name = s.model_name,
          t.workspace_id = s.workspace_id,
          t.dataset_id = s.dataset_id,
          t.workspace_name = s.workspace_name,
          t.dataset_name = s.dataset_name,
          t.result_value = s.result_value,
          t.baseline_model = s.baseline_model,
          t.baseline_value = s.baseline_value,
          t.absolute_delta = s.absolute_delta,
          t.delta_pct = s.delta_pct,
          t.fail_delta_pct_threshold = s.fail_delta_pct_threshold,
          t.status = s.status,
          t.error_message = s.error_message
        WHEN NOT MATCHED THEN INSERT (
          run_id, run_date, run_timestamp, check_name, model_name,
          workspace_id, dataset_id, workspace_name, dataset_name,
          result_value, baseline_model, baseline_value, absolute_delta,
          delta_pct, fail_delta_pct_threshold, status, error_message
        ) VALUES (
          s.run_id, s.run_date, s.run_timestamp, s.check_name, s.model_name,
          s.workspace_id, s.dataset_id, s.workspace_name, s.dataset_name,
          s.result_value, s.baseline_model, s.baseline_value, s.absolute_delta,
          s.delta_pct, s.fail_delta_pct_threshold, s.status, s.error_message
        )
        """)

    # Preflight connectivity for affected workspace/datasets
    probe_targets = failures_df[["workspace_id", "dataset_id", "workspace_name", "dataset_name"]].drop_duplicates()
    for _, target in probe_targets.iterrows():
        _, probe_err = evaluate_dax_with_retry(
            target["dataset_id"],
            target["workspace_id"],
            'EVALUATE ROW("probe", 1)'
        )
        if probe_err:
            print(f"  Connectivity warning for {target['workspace_name']} / {target['dataset_name']}: {probe_err}")

    # Pre-fetch baseline values with retry
    baselines = {}
    baseline_model_map = {}

    # Static baselines
    for check_name in expected_checks:
        cfg = cfg_map[check_name]
        if cfg["baseline_mode"] == "STATIC":
            baselines[check_name] = float(cfg["static_baseline_value"])
            baseline_model_map[check_name] = "STATIC_VALUE"

    # Model baselines
    for _, b in model_baselines_df.iterrows():
        check_name = b["check_name"]
        if cfg_map.get(check_name, {}).get("baseline_mode") != "MODEL":
            continue

        result_df, error = evaluate_dax_with_retry(
            b["baseline_dataset_id"],
            b["baseline_workspace_id"],
            b["baseline_dax"]
        )
        if error:
            print(f"  Baseline error for {check_name}: {error}")
            baselines[check_name] = None
        else:
            value, extract_error = extract_numeric_value(result_df)
            if extract_error:
                print(f"  Baseline type error for {check_name}: {extract_error}")
                baselines[check_name] = None
            else:
                baselines[check_name] = value

        baseline_model_map[check_name] = b["baseline_model"]

    pass_count = fail_count = error_count = 0
    results_batch = []
    num_rows = len(failures_df)

    for row_idx, (_, row) in enumerate(failures_df.iterrows()):
        check_name = row["check_name"]
        model_name = row["model_name"]
        workspace_id = row["workspace_id"]
        dataset_id = row["dataset_id"]
        workspace = row["workspace_name"]
        dataset = row["dataset_name"]
        dax = row["dax_expression"]
        row_threshold = float(row["fail_delta_pct_threshold"])
        baseline_value = baselines.get(check_name)
        baseline_model = baseline_model_map.get(check_name)

        result_value = absolute_delta = delta_pct = error_msg = None
        status = "PASS"

        result_df, exec_error = evaluate_dax_with_retry(dataset_id, workspace_id, dax)

        if exec_error:
            status = "ERROR"
            error_msg = exec_error[:500]
            error_count += 1
        else:
            value, extract_error = extract_numeric_value(result_df)
            if extract_error:
                status = "ERROR"
                error_msg = extract_error
                error_count += 1
            else:
                result_value = value
                if baseline_value is not None and result_value is not None:
                    absolute_delta = result_value - baseline_value
                    if baseline_value != 0:
                        delta_pct = (absolute_delta / baseline_value) * 100
                        status = "FAIL" if abs(delta_pct) > row_threshold else "PASS"
                    else:
                        status = "FAIL" if result_value != 0 else "PASS"
                elif baseline_value is None:
                    status = "ERROR"
                    error_msg = "Baseline is unavailable"
                    error_count += 1
                else:
                    status = "ERROR"
                    error_msg = "Result value is None"
                    error_count += 1

                if status == "PASS":
                    pass_count += 1
                elif status == "FAIL":
                    fail_count += 1

        results_batch.append({
            "run_id": RUN_ID,
            "run_date": RUN_DATE,
            "run_timestamp": RUN_TIMESTAMP,
            "check_name": check_name,
            "model_name": model_name,
            "workspace_id": workspace_id,
            "dataset_id": dataset_id,
            "workspace_name": workspace,
            "dataset_name": dataset,
            "result_value": result_value,
            "baseline_model": baseline_model,
            "baseline_value": baseline_value,
            "absolute_delta": absolute_delta,
            "delta_pct": delta_pct,
            "fail_delta_pct_threshold": row_threshold,
            "status": status,
            "error_message": error_msg
        })

        if len(results_batch) >= 100 or row_idx == num_rows - 1:
            merge_results_batch(results_batch)
            results_batch = []

    print("\nRerun complete")
    print(f"    PASS:  {pass_count}")
    print(f"    FAIL:  {fail_count}")
    print(f"    ERROR: {error_count}")

## Results

In [ ]:
from IPython.display import display

results = spark.sql(f"""
    SELECT check_name, model_name, result_value, baseline_value, delta_pct, fail_delta_pct_threshold, status, error_message
    FROM {FULL_SCHEMA}.validation_results
    WHERE run_id = '{RUN_ID}'
    ORDER BY check_name, model_name
""").toPandas()

if len(results) > 0:
    display(results)
else:
    print("No results written for this run.")